In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from segmentiq.clean import clean_df
from segmentiq.data import get_data
from segmentiq.rfm import create_rfm_table

In [ ]:
root = Path.cwd().parents[0].resolve()
raw_path = root / Path("data/raw/Online Retail.xlsx")
cache_path = root / Path("data/cache/retail.parquet")
df = get_data(raw_path, cache_path)
cleaned_df = clean_df(df)

In [ ]:
rfm = create_rfm_table(cleaned_df)
rfm.head()

Function works, we can see our desired columns: Recency, Frequency and Monetary. There are 4320 customers and not 4362, because 30 of them have negative net revenue, 7 have zero and 5 more are so close to zero that only floating point error keeps them above it. All of them spent nothing, so they are dropped.

In [ ]:
rfm.info()

In [ ]:
rfm.describe()

We can see that Frequency and Monetary have outliers, and in Frequency the majority of the observations are 1. It means that the business has a lot of people who buy only once. On the other hand, outliers show us that there is a small group of people who order a lot.

In [ ]:
cols = ["Recency", "Frequency", "Monetary"]
for col in cols:
    g = sns.histplot(rfm[col])
    g.figure.suptitle(f"Distribution of the {col}")
    plt.show()

Distributions show us what I said above and even more. To start with, even though Recency is right skewed, it is not as extreme as in the other columns, and this skewness looks normal for client churn: you have many active clients and clients who churned, and they create this tail. Distributions of Frequency and Monetary have a high peak and a big right tail. I will use quantile based cut for these columns in order to not have 99% of the data in one bin and somehow standardize the data. However, I can face a problem when the Frequency column creates two identical bins, because there are a lot of the same values.

In [ ]:
rfm["R"] = pd.qcut(rfm["Recency"], q=4, labels=range(4, 0, -1)).astype(int)
rfm["F"] = pd.cut(
    rfm["Frequency"], bins=[1, 2, 4, 9, np.inf], labels=range(1, 5), right=False
).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"], q=4, labels=range(1, 5)).astype(int)
rfm["RFM_score"] = rfm["R"] + rfm["F"] + rfm["M"]

I was right, the Frequency column gives an error if I use qcut. Instead of qcut I used a simple cut with my own bins. It means that my groups are not equal: 34.6% of the customers have F = 1, while R = 1 and M = 1 cover about 25% each, so the score is a bit shifted down. I keep it this way, because one time buyers are a real segment and I do not want to split them between two bins.

Now I want to create a category for each customer, I will use the bronze, silver and gold gradation. After KMeans in the next steps I will create a profile for each cluster.

In [ ]:
labels_cat = ["Bronze", "Silver", "Gold"]
bins_cat = [0, 5, 9, 12]
rfm["Category"] = pd.cut(rfm["RFM_score"], bins=bins_cat, labels=labels_cat)
rfm.head()

Okay, now we divided our customers into 3 categories and I want to see statistics for each of them.

In [ ]:
rfm["Category"].value_counts()

In [ ]:
rfm.groupby("Category")[["R", "F", "M"]].mean()

In [ ]:
grouped = rfm.groupby("Category")[["Recency", "Frequency", "Monetary"]].mean()
grouped

In [ ]:
grouped_rfm = rfm.groupby("Category")["Monetary"].sum()
grouped_rfm = (grouped_rfm / grouped_rfm.sum() * 100).round(1)
grouped_rfm

I can suggest that Bronze are customers who made 1-2 orders and churned. Silver are average customers, they order a good amount and with normal frequency. Gold are our champions, they have the best RFM. We can see that the most frequent category is Silver, then Bronze and Gold. We can also see that Gold customers generate 72% of the revenue whereas Bronze only 5%. Knowing this, I can suggest that the business should try to keep these Gold customers at any price and focus their promotions on them.